# Train a GenAI-VMS adapter on Kaggle (P5-D2 TG / P5-J2 PhaVR)

QLoRA fine-tuning of `Qwen/Qwen2.5-VL-3B-Instruct` on a dataset built by `ml/training/tg/build_dataset.py`
(TG) or `ml/training/phavr/build_dataset.py` (PhaVR). The same notebook trains both; only the dataset differs.

**Before running** (Settings panel): accelerator **GPU T4 x1** (or P100), **Internet on** (for pip and the model
download). Attach your dataset (`kaggle datasets create` from the builder's output folder, which has
`dataset-metadata.json`) and put `ml/training/train_adapter.py` in the same dataset.

**Status:** the trainer was exercised end to end only on a 10 M-parameter random model (it learns, resumes
exactly, and writes the files `hf_local` loads). This notebook has not been run on Kaggle, and no real
annotation exists yet.

In [ ]:
# Settings: edit these.
DATASET = "/kaggle/input/tg-v1"            # folder with train.jsonl, val.jsonl, frames/, train_adapter.py
NAME = "tg-v1"                              # tg-v1 or phavr-v1
BASE = "Qwen/Qwen2.5-VL-3B-Instruct"
EPOCHS = 3
OUT = f"/kaggle/working/adapters/{NAME}"
RESUME = False   # True after adding a previous run's output as an input (see PREVIOUS below)

In [ ]:
!pip install -q "transformers>=4.49,<5" "peft>=0.13" "bitsandbytes>=0.44" "accelerate>=1.0" pillow
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

In [ ]:
# To continue a run that hit the session limit: add its output as an input and copy the checkpoint back.
import pathlib
import shutil
PREVIOUS = pathlib.Path("/kaggle/input/previous-run") / NAME   # only used when RESUME is True
if RESUME and PREVIOUS.exists():
    shutil.copytree(PREVIOUS, OUT, dirs_exist_ok=True)
    print("restored", [p.name for p in pathlib.Path(OUT).iterdir()])

In [ ]:
import subprocess
import sys
cmd = [sys.executable, f"{DATASET}/train_adapter.py", "--dataset", DATASET, "--out", OUT,
       "--base", BASE, "--epochs", str(EPOCHS), "--save-every", "25"] + (["--resume"] if RESUME else [])
print(" ".join(cmd))
subprocess.run(cmd, check=True)   # prints one JSON line per optimizer step and per epoch

In [ ]:
# The loss curve and the per-epoch validation numbers.
import json
import matplotlib.pyplot as plt
rows = [json.loads(line) for line in open(f"{OUT}/metrics.jsonl")]
steps = [r for r in rows if r["kind"] == "step"]
epochs = [r for r in rows if r["kind"] == "epoch"]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot([r["step"] for r in steps], [r["loss"] for r in steps])
ax[0].set(title="training loss", xlabel="optimizer step")
ax[1].plot([r["epoch"] for r in epochs], [r["val_loss"] for r in epochs], marker="o", label="val loss")
ax[1].plot([r["epoch"] for r in epochs], [r["val_token_accuracy"] for r in epochs], marker="o", label="val token accuracy")
ax[1].set(title="validation", xlabel="epoch")
ax[1].legend()
plt.show()
print(open(f"{OUT}/model_card.md").read())

## Getting the adapter into the system

Download `adapters/<name>/best/` and `model_card.md` from the notebook output, then on a machine that can reach MinIO:

```bash
uv run python ml/training/train_adapter.py --out adapters/tg-v1 --upload-only s3://vms-models/tg/v1
```

Then point `phase_tg` (or `phase_vr`) at it in `config/models.yaml` (`provider: hf_local`,
`base: Qwen/Qwen2.5-VL-3B-Instruct`, `adapter: s3://vms-models/tg/v1`) and score it with
`ml/evaluation/reasoning/phase/evaluate.py` (TG) or `ml/evaluation/reasoning/phavr/evaluate.py` (PhaVR)
**on the test split**, against the zero-shot baseline. Validation loss is not that result.

In [ ]:
# Zip the result so it can be downloaded in one piece.
shutil.make_archive(f"/kaggle/working/{NAME}", "zip", OUT)
print(f"/kaggle/working/{NAME}.zip")